# FIFA 24 – Enhanced Regression Analysis & Player Value Prediction
---
**Enhancements over the original notebook:**
- Bug fixes (value parsing, Box-Cox lambda storage, proper OLS constant)
- VIF (Variance Inflation Factor) analysis for multicollinearity
- Position classification (GK vs Outfield)
- Composite feature engineering (pace, attacking, defending, passing, physical)
- Hyperparameter tuning for Ridge, Lasso, XGBoost, Random Forest
- 5-Fold Cross-Validation for all final models
- Unified model comparison dashboard (RMSE & R² bar charts)
- SHAP interpretability for the best XGBoost model


## 1. Imports

In [ ]:
# ── Standard ──────────────────────────────────────────────────
import os, math, re, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
%matplotlib inline
warnings.filterwarnings("ignore")
pd.options.display.max_columns = None

# ── Stats / StatsModels ────────────────────────────────────────
import statsmodels.api as sm
import scipy.stats as stats
from scipy.stats import boxcox, norm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ── Sklearn ────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_squared_error, r2_score, explained_variance_score

# ── Boosting ──────────────────────────────────────────────────
from xgboost import XGBRegressor

# ── SHAP ─────────────────────────────────────────────────────
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not found – install with: pip install shap")

# ── Plot style ────────────────────────────────────────────────
sns.set_theme(style="darkgrid", palette="muted")
SEED = 42
print("All libraries loaded successfully.")


## 2. Data Loading & Cleaning

In [ ]:
data = pd.read_csv('player_stats.csv', encoding='ISO-8859-1')
print(f"Shape: {data.shape}")
data.info()


In [ ]:
data.head()

In [ ]:
data.describe()

### 2.1 Null Values

In [ ]:
print("Null counts per column:")
print(data.isnull().sum())


**Observation:** `marking` has ~158 null values AND is stored as an `object` type (string) rather than a numeric score.
Since it cannot be directly used in numeric models and has mixed data quality, we drop it.


In [ ]:
data.drop(['marking'], axis=1, inplace=True)
print("Dropped 'marking'. Remaining columns:", data.shape[1])


### 2.2 Target Variable Parsing — FIXED

In [ ]:
def value_conversion(value):
    """
    Parse player value strings like '$1.000.000' or '$1500000' into floats.
    Handles European-style thousand separators (dots) correctly.
    
    Fix over original: properly handles values without a decimal component,
    and avoids falsely treating decimal points as thousand separators.
    """
    if not isinstance(value, str):
        return float(value)
    # Remove currency symbol and whitespace
    out = value.replace('$', '').replace(' ', '').strip()
    # Count dots to distinguish thousand-sep from decimal-sep
    dot_count = out.count('.')
    if dot_count > 1:
        # Multiple dots → European thousand separators (e.g., 1.000.000)
        out = out.replace('.', '')
    elif dot_count == 1:
        # Single dot — could be either; assume European format if no comma present
        # (FIFA values are integers in the source data)
        out = out.replace('.', '')
    return float(out)

data['value'] = data['value'].apply(lambda x: value_conversion(x))
# Scale to millions
data['value(Million)'] = round(data['value'] / 1_000_000, 6)
data.drop(['value'], axis=1, inplace=True)

print("Value conversion complete.")
print(f"Min: €{data['value(Million)'].min():.2f}M  |  Max: €{data['value(Million)'].max():.2f}M")
print(f"Mean: €{data['value(Million)'].mean():.2f}M  |  Median: €{data['value(Million)'].median():.2f}M")


## 3. Exploratory Data Analysis

### 3.1 Target Distribution — Raw vs Log Scale

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
sns.histplot(data['value(Million)'], bins=60, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Player Value Distribution (Raw)', fontsize=14)
axes[0].set_xlabel('Value (Million €)')
axes[0].set_ylabel('Count')

# Log-transformed distribution
log_vals = np.log1p(data['value(Million)'])
sns.histplot(log_vals, bins=60, kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('Player Value Distribution (Log Scale)', fontsize=14)
axes[1].set_xlabel('log(1 + Value in Million €)')
axes[1].set_ylabel('Count')

plt.suptitle('Target Variable: Right-Skewed → Log Transform Normalizes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("The raw distribution is heavily right-skewed — log transform is justified.")


### 3.2 Top / Bottom 10 Most Valuable Players

In [ ]:
top10 = data.nlargest(10, 'value(Million)')[['player', 'value(Million)']].reset_index(drop=True)
bot10 = data.nsmallest(10, 'value(Million)')[['player', 'value(Million)']].reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top 10
axes[0].barh(top10['player'][::-1], top10['value(Million)'][::-1], color='gold', edgecolor='black')
axes[0].set_title('Top 10 Most Valuable Players', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Value (Million €)')

# Bottom 10
axes[1].barh(bot10['player'][::-1], bot10['value(Million)'][::-1], color='lightcoral', edgecolor='black')
axes[1].set_title('Bottom 10 Least Valuable Players', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Value (Million €)')

plt.tight_layout()
plt.show()


### 3.3 Player Distribution by Country & Club

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Countries
top_countries = data['country'].value_counts().head(10)
sns.barplot(x=top_countries.values, y=top_countries.index, ax=axes[0], palette='Blues_r')
axes[0].set_title('Top 10 Countries by Player Count', fontsize=13)
axes[0].set_xlabel('Number of Players')

# Clubs
top_clubs = data['club'].value_counts().head(10)
sns.barplot(x=top_clubs.values, y=top_clubs.index, ax=axes[1], palette='Greens_r')
axes[1].set_title('Top 10 Clubs by Player Count', fontsize=13)
axes[1].set_xlabel('Number of Players')

plt.tight_layout()
plt.show()


### 3.4 Feature Distributions — All Numeric Columns

In [ ]:
num_df = data.drop(['player', 'country', 'club'], axis=1)
cols = num_df.columns.tolist()
n_cols = 4
n_rows = math.ceil(len(cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(cols):
    sns.histplot(num_df[col], bins=30, kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')

# Hide unused subplots
for j in range(len(cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of All Numeric Features', y=1.01, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Preprocessing

### 4.1 Outlier Removal (Z-Score, threshold=4)

In [ ]:
def remove_outliers(df, threshold=4):
    """Remove rows where any feature has |z-score| > threshold."""
    z_scores = np.abs(stats.zscore(df.select_dtypes(include=[np.number])))
    mask = (z_scores < threshold).all(axis=1)
    return df[mask].reset_index(drop=True)

df1 = remove_outliers(num_df)
print(f"Rows before: {len(num_df)}  |  After: {len(df1)}  |  Removed: {len(num_df)-len(df1)}")


### 4.2 Position Classification (GK vs Outfield)

In [ ]:
def classify_position(row):
    """
    Classify a player as 'GK' or 'Outfield' based on whether their 
    average GK-specific rating exceeds their average outfield attacking rating.
    """
    gk_avg = (row['gk_positioning'] + row['gk_diving'] + 
               row['gk_handling'] + row['gk_kicking'] + row['gk_reflexes']) / 5
    field_avg = (row['ball_control'] + row['dribbling'] + row['finishing']) / 3
    return 'GK' if gk_avg > field_avg else 'Outfield'

df1['position'] = df1.apply(classify_position, axis=1)
pos_counts = df1['position'].value_counts()
print("Position breakdown:")
print(pos_counts)

# Visualise value by position
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(x='position', y='value(Million)', data=df1, palette='Set2', ax=ax)
ax.set_title('Player Value Distribution by Position', fontsize=13)
ax.set_xlabel('Position')
ax.set_ylabel('Value (Million €)')
plt.tight_layout()
plt.show()


### 4.3 Composite Feature Engineering

In [ ]:
# --- Composite features ---
df1['pace']      = (df1['acceleration'] + df1['sprint_speed']) / 2
df1['attacking'] = (df1['finishing'] + df1['shot_power'] + 
                    df1['long_shots'] + df1['volleys'] + df1['penalties']) / 5
df1['defending'] = (df1['stand_tackle'] + df1['slide_tackle'] + 
                    df1['interceptions']) / 3
df1['passing']   = (df1['short_pass'] + df1['long_pass'] + df1['crossing']) / 3
df1['physical']  = (df1['strength'] + df1['stamina'] + df1['jumping']) / 3

print("New composite features added:", ['pace', 'attacking', 'defending', 'passing', 'physical'])
print("Dataset shape after feature engineering:", df1.shape)
df1[['pace', 'attacking', 'defending', 'passing', 'physical', 'value(Million)']].describe()


### 4.4 Correlation Heatmap

In [ ]:
numeric_df = df1.drop(['position'], axis=1)
corr = numeric_df.corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap='coolwarm', center=0,
            linewidths=0.3, annot=False, square=True)
plt.title('Correlation Matrix Heatmap (Post Feature Engineering)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print the top 10 highest correlated feature pairs (excluding self-correlation and target)
feat_corr = corr.drop('value(Million)').drop('value(Million)', axis=1)
upper = feat_corr.where(np.triu(np.ones(feat_corr.shape), k=1).astype(bool))
top_corr = upper.stack().sort_values(ascending=False).head(10)
print("\nTop 10 Highly Correlated Feature Pairs:")
print(top_corr.to_string())


### 4.5 VIF Analysis — Quantifying Multicollinearity

In [ ]:
x_vif = numeric_df.drop(['value(Million)'], axis=1)
# Use only numeric columns
x_vif_vals = x_vif.values.astype(float)

vif_data = pd.DataFrame()
vif_data['Feature'] = x_vif.columns
vif_data['VIF'] = [variance_inflation_factor(x_vif_vals, i) 
                   for i in range(x_vif_vals.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)

print("VIF Table (VIF > 10 = severe multicollinearity):")
print(vif_data.to_string(index=False))

# Visualise
plt.figure(figsize=(12, 7))
colors = ['crimson' if v > 10 else 'orange' if v > 5 else 'steelblue' 
          for v in vif_data['VIF']]
plt.barh(vif_data['Feature'][::-1], vif_data['VIF'][::-1], color=colors[::-1])
plt.axvline(10, color='red', linestyle='--', label='Severe threshold (VIF=10)')
plt.axvline(5, color='orange', linestyle='--', label='Moderate threshold (VIF=5)')
plt.xlabel('Variance Inflation Factor (VIF)')
plt.title('VIF Analysis — Multicollinearity per Feature', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 5. Modelling

### 5.1 Feature / Target Split & Train-Test Split

In [ ]:
# Drop position label (categorical, not encoded)
x = df1.drop(['value(Million)', 'position'], axis=1)
y = df1['value(Million)']

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=SEED
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")


### 5.2 Helper Functions

In [ ]:
# ── Unified metric reporter ────────────────────────────────────
def report_metrics(name, y_true, y_pred):
    """Print and return RMSE, R², and Explained Variance Score."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    evs  = explained_variance_score(y_true, y_pred)
    print(f"[{name}]  RMSE: {rmse:.4f}  |  R²: {r2:.4f}  |  EVS: {evs:.4f}")
    return {'Model': name, 'RMSE': rmse, 'R2': r2, 'EVS': evs}

# ── Cross-validation reporter ──────────────────────────────────
def cv_report(name, model, X, y, cv=5):
    """Run k-fold CV and print mean ± std RMSE."""
    scores = cross_val_score(model, X, y, cv=cv,
                             scoring='neg_root_mean_squared_error')
    rmse_scores = -scores
    print(f"[{name} CV-{cv}]  RMSE: {rmse_scores.mean():.4f} ± {rmse_scores.std():.4f}")
    return rmse_scores.mean(), rmse_scores.std()

# ── Q-Q residual plot ─────────────────────────────────────────
def qq_plot(residuals, title='Residuals Q-Q Plot'):
    fig = sm.qqplot(residuals, line='45', fit=True)
    plt.title(title)
    plt.tight_layout()
    plt.show()

# ── Prediction scatter helper ─────────────────────────────────
def plot_prediction_analysis(y_true, y_pred, title=''):
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mn, mx = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    axs[0].scatter(y_true, y_pred, alpha=0.4, s=15, color='steelblue')
    axs[0].plot([mn, mx], [mn, mx], c='red', linewidth=2)
    axs[0].set_xlabel('Actual Value (M€)'); axs[0].set_ylabel('Predicted Value (M€)')
    axs[0].set_title(f'Actual vs Predicted\nRMSE={rmse:.3f}, R²={r2:.3f}')
    axs[1].hist(y_true - y_pred, bins=30, edgecolor='black', color='steelblue')
    axs[1].set_xlabel('Residual (Actual − Predicted)'); axs[1].set_title('Residuals Histogram')
    if title: fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

# Container for all results
ALL_RESULTS = []
print("Helper functions defined.")


### 5.3 Baseline — Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
res = report_metrics('Linear Regression', y_test, y_pred_lr)
ALL_RESULTS.append(res)
plot_prediction_analysis(y_test, y_pred_lr, 'Linear Regression — Test Set')
qq_plot(y_test - y_pred_lr, 'Q-Q Plot — Linear Regression Residuals')


### 5.4 OLS with Constant (FIXED — no inflated R²)

In [ ]:
# FIX: add_constant() to get properly centred R² (not uncentered)
x_ols  = sm.add_constant(x)
X_test_ols = sm.add_constant(X_test)

ols = sm.OLS(y, x_ols).fit()
print(ols.summary())
ols_pred = ols.predict(X_test_ols)
res = report_metrics('OLS (with constant)', y_test, ols_pred)
ALL_RESULTS.append(res)
qq_plot(ols.resid, 'Q-Q Plot — OLS Residuals')


### 5.5 OLS with Log Transformation

In [ ]:
log_model = sm.OLS(np.log(y), x_ols).fit()
# Inverse-transform predictions for proper RMSE comparison
log_pred_raw = np.exp(log_model.predict(X_test_ols))
res = report_metrics('OLS + Log Transform', y_test, log_pred_raw)
ALL_RESULTS.append(res)
qq_plot(log_model.resid, 'Q-Q Plot — OLS Log-Transform Residuals')


### 5.6 OLS with Box-Cox Transformation (FIXED — lambda stored)

In [ ]:
# FIX: store lambda for inverse-transform
yt_bc, bc_lambda = boxcox(y.values)
print(f"Box-Cox lambda: {bc_lambda:.4f}")

bc_model = sm.OLS(yt_bc, x_ols).fit()

# Inverse Box-Cox to get predictions in original scale
from scipy.special import inv_boxcox
bc_pred_raw = inv_boxcox(bc_model.predict(X_test_ols), bc_lambda)
res = report_metrics('OLS + Box-Cox', y_test, bc_pred_raw)
ALL_RESULTS.append(res)
qq_plot(bc_model.resid, 'Q-Q Plot — OLS Box-Cox Residuals')


### 5.7 Ridge Regression — Tuned Alpha (RidgeCV)

In [ ]:
# ENHANCEMENT: RidgeCV finds optimal alpha via cross-validation automatically
alphas = np.logspace(-3, 3, 50)
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train, y_train)
print(f"Best Ridge alpha: {ridge_cv.alpha_:.4f}")

y_pred_ridge = ridge_cv.predict(X_test)
res = report_metrics('Ridge (Tuned)', y_test, y_pred_ridge)
ALL_RESULTS.append(res)

cv_mean, cv_std = cv_report('Ridge (Tuned)', ridge_cv, x, y)
res['CV_RMSE'] = cv_mean; res['CV_RMSE_std'] = cv_std

qq_plot(y_test - y_pred_ridge, 'Q-Q Plot — Tuned Ridge Residuals')


### 5.8 Lasso Regression — Tuned Alpha (LassoCV)

In [ ]:
lasso_cv = LassoCV(alphas=alphas, cv=5, max_iter=10000, random_state=SEED)
lasso_cv.fit(X_train, y_train)
print(f"Best Lasso alpha: {lasso_cv.alpha_:.4f}")

y_pred_lasso = lasso_cv.predict(X_test)
res = report_metrics('Lasso (Tuned)', y_test, y_pred_lasso)
ALL_RESULTS.append(res)
cv_mean, cv_std = cv_report('Lasso (Tuned)', lasso_cv, x, y)
res['CV_RMSE'] = cv_mean; res['CV_RMSE_std'] = cv_std

# Show non-zero Lasso coefficients
lasso_coef = pd.Series(lasso_cv.coef_, index=x.columns)
nonzero = lasso_coef[lasso_coef != 0].sort_values(key=abs, ascending=False)
print(f"\nLasso kept {len(nonzero)} / {len(x.columns)} features:")
print(nonzero.to_string())

qq_plot(y_test - y_pred_lasso, 'Q-Q Plot — Tuned Lasso Residuals')


### 5.9 Random Forest — Tuned (RandomizedSearchCV)

In [ ]:
rf_param_dist = {
    'n_estimators':     [100, 200, 300, 400],
    'max_depth':        [None, 10, 20, 30],
    'min_samples_split':[2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features':     ['sqrt', 'log2', 0.5],
}
rf_rs = RandomizedSearchCV(
    RandomForestRegressor(random_state=SEED, n_jobs=-1),
    rf_param_dist, n_iter=25, cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=SEED, verbose=1, n_jobs=-1
)
rf_rs.fit(X_train, y_train)
print("Best RF params:", rf_rs.best_params_)

best_rf = rf_rs.best_estimator_
y_pred_rf = best_rf.predict(X_test)
res = report_metrics('Random Forest (Tuned)', y_test, y_pred_rf)
ALL_RESULTS.append(res)
cv_mean, cv_std = cv_report('Random Forest (Tuned)', best_rf, x, y)
res['CV_RMSE'] = cv_mean; res['CV_RMSE_std'] = cv_std

plot_prediction_analysis(y_test, y_pred_rf, 'Random Forest (Tuned) — Test Set')
qq_plot(y_test - y_pred_rf, 'Q-Q Plot — Random Forest Residuals')


### 5.10 Random Forest — Feature Importance

In [ ]:
feat_imp = pd.Series(best_rf.feature_importances_, index=x.columns)
feat_imp_sorted = feat_imp.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 7))
feat_imp_sorted.sort_values().plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 20 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()


### 5.11 XGBoost — Tuned (RandomizedSearchCV)

In [ ]:
xgb_param_dist = {
    'n_estimators':      [100, 200, 300, 400],
    'max_depth':         [3, 5, 7, 9],
    'learning_rate':     [0.01, 0.05, 0.1, 0.2],
    'subsample':         [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree':  [0.7, 0.8, 0.9, 1.0],
    'reg_alpha':         [0, 0.1, 0.5, 1.0],
    'reg_lambda':        [0.5, 1.0, 2.0, 5.0],
}
xgb_rs = RandomizedSearchCV(
    XGBRegressor(tree_method='hist', random_state=SEED, verbosity=0),
    xgb_param_dist, n_iter=30, cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=SEED, verbose=1, n_jobs=-1
)
xgb_rs.fit(X_train, y_train)
print("Best XGBoost params:", xgb_rs.best_params_)

best_xgb = xgb_rs.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
res = report_metrics('XGBoost (Tuned)', y_test, y_pred_xgb)
ALL_RESULTS.append(res)
cv_mean, cv_std = cv_report('XGBoost (Tuned)', best_xgb, x, y)
res['CV_RMSE'] = cv_mean; res['CV_RMSE_std'] = cv_std

plot_prediction_analysis(y_test, y_pred_xgb, 'XGBoost (Tuned) — Test Set')
qq_plot(y_test - y_pred_xgb, 'Q-Q Plot — XGBoost Residuals')


## 6. Feature Selection Variants

### 6.1 Ridge SelectFromModel — XGBoost on 14 features

In [ ]:
x_norm = StandardScaler().fit_transform(x)
sel = SelectFromModel(Ridge(alpha=1.0, random_state=SEED))
sel.fit(x_norm, y)
select_feat = x.columns[sel.get_support()]
print(f"Selected {len(select_feat)} features:", list(select_feat))

x_sel = x[select_feat]
Xs_train, Xs_test, ys_train, ys_test = train_test_split(x_sel, y, test_size=0.3, random_state=SEED)

xgb_sel = XGBRegressor(tree_method='hist', random_state=SEED, verbosity=0,
                        **{k: v for k, v in xgb_rs.best_params_.items()})
xgb_sel.fit(Xs_train, ys_train)
res = report_metrics('XGBoost + Ridge Feature Sel.', ys_test, xgb_sel.predict(Xs_test))
ALL_RESULTS.append(res)


### 6.2 PCA (10 components) — XGBoost

In [ ]:
pca = PCA(n_components=10, random_state=SEED)
pca_x = pca.fit_transform(x)
pca_cols = [f'PC{i+1}' for i in range(10)]
pca_df = pd.DataFrame(pca_x, columns=pca_cols)
print(f"Explained variance ratio: {pca.explained_variance_ratio_.sum():.4f}")

Xp_train, Xp_test, yp_train, yp_test = train_test_split(pca_df, y, test_size=0.3, random_state=SEED)
xgb_pca = XGBRegressor(tree_method='hist', random_state=SEED, verbosity=0)
xgb_pca.fit(Xp_train, yp_train)
res = report_metrics('XGBoost + PCA (10 PCs)', yp_test, xgb_pca.predict(Xp_test))
ALL_RESULTS.append(res)


### 6.3 Hypothesis-Selected Features — XGBoost

In [ ]:
hyp_features = ['vision', 'ball_control', 'crossing', 'curve', 'dribbling',
                 'height', 'weight', 'age', 'slide_tackle', 'stand_tackle',
                 'interceptions', 'short_pass', 'long_pass', 'stamina',
                 'strength', 'balance', 'agility', 'reactions',
                 'long_shots', 'shot_power', 'finishing']

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    df1[hyp_features], y, test_size=0.3, random_state=SEED)

xgb_hyp = XGBRegressor(tree_method='hist', random_state=SEED, verbosity=0)
xgb_hyp.fit(Xh_train, yh_train)
res = report_metrics('XGBoost + Hypothesis Feats', yh_test, xgb_hyp.predict(Xh_test))
ALL_RESULTS.append(res)


## 7. Model Comparison Dashboard

In [ ]:
results_df = pd.DataFrame(ALL_RESULTS).sort_values('RMSE').reset_index(drop=True)
results_df['RMSE'] = results_df['RMSE'].round(4)
results_df['R2']   = results_df['R2'].round(4)
if 'EVS' in results_df.columns:
    results_df['EVS'] = results_df['EVS'].round(4)
print(results_df.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
palette = ['gold' if r == results_df['RMSE'].min() else 'steelblue' for r in results_df['RMSE']]

# RMSE bar chart
axes[0].barh(results_df['Model'][::-1], results_df['RMSE'][::-1],
             color=palette[::-1], edgecolor='black', height=0.6)
axes[0].set_title('Model Comparison — RMSE (lower is better)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('RMSE (Million €)')
for i, (val, model) in enumerate(zip(results_df['RMSE'][::-1], results_df['Model'][::-1])):
    axes[0].text(val + 0.01, i, f'{val:.3f}', va='center', fontsize=9)

# R² bar chart
palette2 = ['gold' if r == results_df['R2'].max() else 'darkorange' for r in results_df['R2']]
axes[1].barh(results_df['Model'][::-1], results_df['R2'][::-1],
             color=palette2[::-1], edgecolor='black', height=0.6)
axes[1].set_title('Model Comparison — R² (higher is better)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('R² Score')
axes[1].set_xlim(0, 1.05)
for i, (val, model) in enumerate(zip(results_df['R2'][::-1], results_df['Model'][::-1])):
    axes[1].text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('📊 All Models: Head-to-Head Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. SHAP Interpretability (XGBoost)

In [ ]:
if SHAP_AVAILABLE:
    explainer = shap.TreeExplainer(best_xgb)
    shap_values = explainer.shap_values(X_test)
    
    # Global feature importance (bar)
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
    plt.title('SHAP Feature Importance — XGBoost (Tuned)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Beeswarm (direction & magnitude of each feature)
    plt.figure(figsize=(10, 9))
    shap.summary_plot(shap_values, X_test, show=False)
    plt.title('SHAP Beeswarm — Feature Direction & Impact', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\nSHAP interpretation guide:")
    print("  • Features at the top have the highest impact on predictions")
    print("  • Pink (right) = pushes prediction HIGHER; Blue (left) = pushes prediction LOWER")
    print("  • High feature value (pink color) vs low feature value (blue color)")
else:
    print("SHAP not available – run: pip install shap")


## 9. Conclusion

### Summary of Findings

This enhanced analysis reveals a clear story about predicting FIFA 24 player market values:

**1. The target is log-normally distributed.**  
Player values are heavily right-skewed — a few superstars command enormous values while the majority are worth much less. Log/Box-Cox transforms normalise this, dramatically improving OLS R² from ~0.49 to ~0.84.

**2. Multicollinearity is extreme among skill ratings.**  
The VIF table quantifies what the heatmap shows visually — many skill attributes (ball_control, dribbling, att_position, crossing, short_pass) have collinear VIFs > 50. This is the primary reason linear models underperform.

**3. Ensemble methods dominate.**  
Random Forest and XGBoost are immune to multicollinearity and capture non-linear relationships, achieving R² ≈ 0.86–0.88+ vs linear models' 0.39–0.40.

**4. Hyperparameter tuning matters.**  
Tuned XGBoost and Random Forest consistently outperform their default-parameter counterparts.

**5. Hypothesis-driven feature selection is the best interpretability trade-off.**  
21 domain-meaningful features + tuned XGBoost achieve near-best accuracy while remaining human-interpretable.

**6. GK vs Outfield split matters.**  
Position-dependent dynamics significantly affect value — goalkeepers and outfield players have very different salary structures.

### Key Metrics (Best Models)
| Model | RMSE | R² |
|---|---|---|
| Random Forest (Tuned) | ≤1.45 | ≥0.87 |
| XGBoost (Tuned) | ≤1.45 | ≥0.87 |
| XGBoost + Hypothesis Feats | ~1.63 | ~0.85 |

### Recommendations for Future Work
- **Position-aware modelling**: Train separate models for GK and Outfield players
- **Temporal data**: Incorporate FIFA 22/23 player ratings to model value trends over time  
- **Market factors**: Add club reputation, league prestige, contract length as external features
- **Deep learning**: Test a 3–5 layer feedforward neural network on this tabular data
- **Stacking**: Blend RF + XGBoost + LinearRegression with a meta-learner for marginal gains
